# FactSet API Trial — Sections 1–4

This notebook is the thin, reproducible front door to the trial. API request construction, transport, classification, catalog parsing, persistence, and entitlement rendering live in `lasr.data.providers.factset`; the notebook only configures and displays them.

## 1. Scope and limitations

Sections 1–4 cover scope, documentation/entitlement evidence, environment/configuration, and API health. Estimates API results are a labeled NON-PIT sensitivity arm. RBICS and Benchmarks are effective-dated but excluded from the strict PIT-safe headline until vendor knowledge-time evidence exists. Async-batch live surfaces remain disabled until VF-FS010-3/RT-FS010-4 is fixed. Sections 5–18 are intentionally owned by later goals.

## 2. Documentation and entitlement summary

The normative offline capability matrix is `docs/factset/capability/MANIFEST.md`; timestamped OBSERVED_LIVE evidence is `docs/factset/entitlements.md`. Entitlement is endpoint/request/time specific: a successful family probe does not imply every operation or identifier type is entitled.

In [ ]:
from __future__ import annotations

import os
import tempfile
from datetime import UTC, datetime
from pathlib import Path

from lasr.data.providers.factset.config import load_trial_config
from lasr.data.providers.factset.discovery import (
    FAMILY_OPERATION_TOTALS,
    run_discovery,
)

## 3. Environment and configuration

`LIVE_PULL` defaults to `False`. Replay constructs no network sender. To opt into live mode, set `LIVE_PULL=True` deliberately and provide `FACTSET_LIVE=1`, `FACTSET_TRIAL_DATA_ROOT`, `FACTSET_USERNAME`, and `FACTSET_API_KEY` outside the notebook. Secret values are never displayed or loaded from files here.

In [ ]:
LIVE_PULL = False


def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "configs/factset/trial.yaml").is_file():
            return candidate
    raise RuntimeError("run this notebook from inside the LASR repository")


REPO_ROOT = _find_repo_root(Path.cwd().resolve())
CONFIG_PATH = REPO_ROOT / "configs/factset/trial.yaml"
CONFIG = load_trial_config(CONFIG_PATH)

configured_root = os.environ.get("FACTSET_TRIAL_DATA_ROOT")
if configured_root:
    CACHE_ROOT = Path(configured_root).expanduser().resolve() / "raw"
else:
    CACHE_ROOT = Path(tempfile.mkdtemp(prefix="fs024-replay-")) / "raw"

print(
    {
        "config_id": CONFIG.config_id,
        "live_pull": LIVE_PULL,
        "families": sorted(CONFIG.families),
        "daily_budget": CONFIG.transport.max_live_calls_per_day,
        "replay_cache": str(CACHE_ROOT),
    }
)

## 4. API health and entitlement evidence

The same deterministic 17-probe plan runs through the FS010 transport. The three subscription-gated Symbology outputs are separate request identities. Replay hits immutable captures or reports `Not captured`; it never turns cached errors into data, and cached account HTTP 401 evidence aborts the health check.

In [ ]:
REPORT = run_discovery(
    config_path=CONFIG_PATH,
    environ=os.environ,
    repo_root=REPO_ROOT,
    code_revision=os.environ.get("FACTSET_CODE_REVISION", "notebook-replay"),
    now=datetime.now(UTC),
    run_id="fs024-notebook-health",
    live=LIVE_PULL,
    cache_root=None if LIVE_PULL else CACHE_ROOT,
    write_outputs=False,
)

for family, operation_total in FAMILY_OPERATION_TOTALS.items():
    family_results = [p for p in REPORT.probes if p.spec.family == family]
    classes = sorted({p.classification.value for p in family_results})
    print(
        f"{family}: probes={len(family_results)}, "
        f"manifest_ops={operation_total}, classes={classes}"
    )

catalog_counts = {
    "fundamentals_non_pit": (
        REPORT.fundamentals_non_pit_summary.total
        if REPORT.fundamentals_non_pit_summary
        else None
    ),
    "fundamentals_pit": (
        REPORT.fundamentals_pit_summary.total
        if REPORT.fundamentals_pit_summary
        else None
    ),
    "estimates": REPORT.estimates_summary.total if REPORT.estimates_summary else None,
}
print(
    {
        "live_calls": REPORT.live_calls,
        "cache_hits": REPORT.cache_hits,
        "catalog_counts": catalog_counts,
        "deferred_async_operations": len(REPORT.deferred),
    }
)
assert REPORT.live_calls == 0 if not LIVE_PULL else True